# Zadanie Tier 2 - Rozproszony System Plików


In [1]:
import os
import ray
import asyncio
import uuid
import time
import random

RAY_ADDRESS = "ray://ray-head:10001"

if ray.is_initialized():
    ray.shutdown()

print(f"Connecting to Ray cluster at {RAY_ADDRESS}...")
ray.init(address=RAY_ADDRESS, namespace="hdfs-simulation")
print("Connected successfully!")
print("Cluster resources:", ray.cluster_resources())

2026-06-01 01:58:33,786	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver


Connecting to Ray cluster at ray://ray-head:10001...
Connected successfully!
Cluster resources: {'node:172.20.0.5': 1.0, 'node:172.20.0.4': 1.0, 'node:__internal_head__': 1.0, 'node:172.20.0.2': 1.0, 'CPU': 8.0, 'object_store_memory': 13002570546.0, 'memory': 30339331278.0, 'node:172.20.0.3': 1.0}


## 1. Węzeł przechowujący - DataNode

Prosty aktor, który trzyma słownik z przypisanymi do niego blokami. Posiada też funkcję do pobrania bloku od innego węzła, co przydaje się przy odtwarzaniu zawartości (po tym jak np. w clusterze jakiś węzeł wyzionął ducha).


In [2]:
@ray.remote
class DataNode:
    def __init__(self, node_id):
        self.node_id = node_id
        self.blocks = {}  # tu trzymamy bloki (blok_id: dane)

    def put_block(self, block_id, data):
        self.blocks[block_id] = data
        return True

    def get_block(self, block_id):
        return self.blocks.get(block_id, None)

    def delete_block(self, block_id):
        if block_id in self.blocks:
            del self.blocks[block_id]
            return True
        return False

    def list_blocks(self):
        return list(self.blocks.keys())

    def ping(self):
        return self.node_id

    def get_id(self):
        return self.node_id

    async def replicate_from(self, block_id, source_datanode):
        data = await source_datanode.get_block.remote(block_id)
        if data:
            self.blocks[block_id] = data
            return True
        return False

## 2. Płyta główna systemu - NameNode

Zarządza metadanymi - wie na jakich węzłach co leży, dba o replikacje i poprawny stan całego zoo.
Jako dodatek post-zajęciowy napisałem go jako aktora **asynchronicznego** wbudowanym w Ray słowem kluczowym `async def`. Dzięki temu wewnątrz Ray'a działa klasyczna pętla `asyncio`. Sprawia to, że diagnoza np. 100 węzłów (`repair_blocks`) leci równolegle, nie zamrażając całego menadżera długimi timeoutami na zepsutych klastrach.


In [3]:
@ray.remote
class NameNode:
    def __init__(self, replication_factor=2):
        self.replication_factor = replication_factor
        self.datanodes = []
        self.artifacts = {}

    def register_datanode(self, datanode_handle):
        self.datanodes.append(datanode_handle)
        return len(self.datanodes)

    def allocate_blocks(self, artifact_name, num_blocks):
        if len(self.datanodes) == 0:
            raise Exception("No DataNodes available for allocation")

        blocks = []
        for _ in range(num_blocks):
            block_id = str(uuid.uuid4())
            replica_nodes = random.sample(
                self.datanodes, min(self.replication_factor, len(self.datanodes))
            )
            blocks.append({"block_id": block_id, "nodes": replica_nodes})

        self.artifacts[artifact_name] = {"blocks": blocks}
        return blocks

    def update_single_block_allocation(self, artifact_name, block_index):
        block_id = str(uuid.uuid4())
        replica_nodes = random.sample(
            self.datanodes, min(self.replication_factor, len(self.datanodes))
        )

        old_block_info = self.artifacts[artifact_name]["blocks"][block_index]

        self.artifacts[artifact_name]["blocks"][block_index] = {
            "block_id": block_id,
            "nodes": replica_nodes,
        }
        return self.artifacts[artifact_name]["blocks"][block_index], old_block_info

    def get_artifact_metadata(self, artifact_name):
        return self.artifacts.get(artifact_name, None)

    def list_artifacts(self):
        return list(self.artifacts.keys())

    def delete_artifact(self, artifact_name):
        metadata = self.artifacts.pop(artifact_name, None)
        return metadata

    async def repair_blocks(self):
        alive_nodes = []
        dead_nodes = []

        async def check_node(dn):
            try:
                await asyncio.wait_for(
                    asyncio.wrap_future(dn.ping.remote().as_future()), timeout=2.0
                )
                return dn, True
            except Exception:
                return dn, False

        results = await asyncio.gather(*(check_node(dn) for dn in self.datanodes))
        for dn, is_alive in results:
            if is_alive:
                alive_nodes.append(dn)
            else:
                dead_nodes.append(dn)

        for dead in dead_nodes:
            if dead in self.datanodes:
                self.datanodes.remove(dead)

        jobs_to_repair = 0
        for art_name, data in self.artifacts.items():
            for block_meta in data["blocks"]:
                alive_replicas = [
                    node for node in block_meta["nodes"] if node in alive_nodes
                ]

                if (
                    len(alive_replicas) < self.replication_factor
                    and len(alive_replicas) > 0
                ):
                    needed = self.replication_factor - len(alive_replicas)
                    available_targets = [
                        n for n in alive_nodes if n not in alive_replicas
                    ]

                    targets = random.sample(
                        available_targets, min(needed, len(available_targets))
                    )
                    source_dn = alive_replicas[0]

                    for target in targets:
                        target.replicate_from.remote(block_meta["block_id"], source_dn)
                        block_meta["nodes"].append(target)
                        jobs_to_repair += 1

                block_meta["nodes"] = block_meta["nodes"].copy()
                for dead in dead_nodes:
                    if dead in block_meta["nodes"]:
                        block_meta["nodes"].remove(dead)

        return jobs_to_repair, len(dead_nodes)

## 3. Klient HDFS

Uzupełniająca klasa - robi za usera. Nie stawiałem z klienta pełnoprawnego aktora dla uproszczenia notatnika. Klient komunikuje się z NameNodem żeby uzyskać "pozwolenie i adres", a potem mieli pliki na chunki i posyła na wyznaczone adresy do the DataNodes.


In [4]:
class HDFSClient:
    def __init__(self, namenode, chunk_size=20):
        self.namenode = namenode
        self.chunk_size = chunk_size

    def create_artifact(self, name, data_string):
        chunks = [
            data_string[i : i + self.chunk_size]
            for i in range(0, len(data_string), self.chunk_size)
        ]

        blocks_meta = ray.get(self.namenode.allocate_blocks.remote(name, len(chunks)))

        write_futures = []
        for idx, chunk in enumerate(chunks):
            block_id = blocks_meta[idx]["block_id"]
            for dn_handle in blocks_meta[idx]["nodes"]:
                write_futures.append(dn_handle.put_block.remote(block_id, chunk))

        ray.get(write_futures)
        print(f"[Client] Saved artifact '{name}' divided into {len(chunks)} blocks.")

    def read_artifact(self, name):
        meta = ray.get(self.namenode.get_artifact_metadata.remote(name))
        if not meta:
            return None

        result = ""
        for block_meta in meta["blocks"]:
            block_id = block_meta["block_id"]

            chunk_data = None
            for dn_handle in block_meta["nodes"]:
                try:
                    chunk_data = ray.get(
                        dn_handle.get_block.remote(block_id), timeout=5.0
                    )
                    if chunk_data is not None:
                        break
                except Exception as e:
                    print(
                        f"[Client] Error downloading block {block_id} from node {dn_handle}: {e}"
                    )
                    pass

            if chunk_data is None:
                raise Exception(
                    f"[Client] Incomplete file hit in: {name}, missing block_id: {block_id}"
                )
            result += chunk_data
        return result

    def update_artifact_chunk(self, name, chunk_index, new_chunk_data):
        new_meta, old_meta = ray.get(
            self.namenode.update_single_block_allocation.remote(name, chunk_index)
        )

        write_futures = []
        for dn_handle in new_meta["nodes"]:
            write_futures.append(
                dn_handle.put_block.remote(new_meta["block_id"], new_chunk_data)
            )

        for dn_handle in old_meta["nodes"]:
            dn_handle.delete_block.remote(old_meta["block_id"])

        ray.get(write_futures)
        print(f"[Client] Replaced chunk at index {chunk_index} in artifact '{name}'.")

    def delete_artifact(self, name):
        meta = ray.get(self.namenode.delete_artifact.remote(name))
        if not meta:
            return False

        # daj znac kazdemu node'owi ktory trzyma kawalek zeby sobie go wyczyscil RAM
        for block_meta in meta["blocks"]:
            for dn_handle in block_meta["nodes"]:
                dn_handle.delete_block.remote(block_meta["block_id"])
        print(f"[Client] File '{name}' completely deleted.")

## 4. Testy i Walidacja systemu


In [5]:
# Inicjalizacja Klastra
namenode = NameNode.remote(replication_factor=2)
datanodes = [DataNode.remote(f"DN-{i}") for i in range(1, 5)]

for dn in datanodes:
    ray.get(namenode.register_datanode.remote(dn))

print("System initialized. Client instances created.")
client = HDFSClient(namenode, chunk_size=30)

System initialized. Client instances created.


In [6]:
# TEST 1: Stworzenie, Odczyt i Wylistowanie Danych z Artefaktu
SECRET_LONG_DOC = (
    "To jest bardzo dlugi string symulujacy zawartosc pliku, ktory chcemy zapisac w naszym rozproszonym systemie plikow. "
    * 2
)
client.create_artifact("moj_raport.txt", SECRET_LONG_DOC)

print("\n--- READING FILE ---")
content = client.read_artifact("moj_raport.txt")
print(f"Read: {content}")
assert content == SECRET_LONG_DOC

metadata = ray.get(namenode.get_artifact_metadata.remote("moj_raport.txt"))
print("\n--- BLOCK MAPPING (As seen by NameNode) ---")
for idx, b in enumerate(metadata["blocks"]):
    node_ids = ray.get([n.get_id.remote() for n in b["nodes"]])
    print(f"[Block {idx} | ID={b['block_id'][:8]}...] located on: {node_ids}")

[Client] Saved artifact 'moj_raport.txt' divided into 8 blocks.

--- READING FILE ---
Read: To jest bardzo dlugi string symulujacy zawartosc pliku, ktory chcemy zapisac w naszym rozproszonym systemie plikow. To jest bardzo dlugi string symulujacy zawartosc pliku, ktory chcemy zapisac w naszym rozproszonym systemie plikow. 

--- BLOCK MAPPING (As seen by NameNode) ---
[Block 0 | ID=7b21a36c...] located on: ['DN-3', 'DN-1']
[Block 1 | ID=10205382...] located on: ['DN-1', 'DN-2']
[Block 2 | ID=2d82880a...] located on: ['DN-1', 'DN-3']
[Block 3 | ID=d0f430ef...] located on: ['DN-4', 'DN-1']
[Block 4 | ID=75e1de68...] located on: ['DN-1', 'DN-3']
[Block 5 | ID=9ab12ecb...] located on: ['DN-3', 'DN-4']
[Block 6 | ID=4c8f7952...] located on: ['DN-3', 'DN-1']
[Block 7 | ID=2c228ece...] located on: ['DN-1', 'DN-2']


In [7]:
# TEST 2: Aaktualizacja
new_chunk_fragment = "##################"
client.update_artifact_chunk("moj_raport.txt", 1, new_chunk_fragment)

print("\n--- NEW READ AFTER UPDATE ---")
content_updated = client.read_artifact("moj_raport.txt")
print(f"Read: {content_updated}")

[Client] Replaced chunk at index 1 in artifact 'moj_raport.txt'.

--- NEW READ AFTER UPDATE ---
Read: To jest bardzo dlugi string sy##################y chcemy zapisac w naszym rozproszonym systemie plikow. To jest bardzo dlugi string symulujacy zawartosc pliku, ktory chcemy zapisac w naszym rozproszonym systemie plikow. 


In [8]:
# TEST 3: Usunięcie i odtworzenie
print("\n--- FAULT TOLERANCE TEST ---")

metadata = ray.get(namenode.get_artifact_metadata.remote("moj_raport.txt"))
unfortunate_node_ref = metadata["blocks"][0]["nodes"][0]
unfortunate_node_id = ray.get(unfortunate_node_ref.get_id.remote())

print(f"FAILURE! Corrupting node: {unfortunate_node_id}")
ray.kill(unfortunate_node_ref)
time.sleep(1)

content_after_crash = client.read_artifact("moj_raport.txt")
print(f"Download during failure success: {content_after_crash[:15]}...")
assert content_updated == content_after_crash

print("Calling RepairBlocks on NameNode...")
jobs_issued, num_dead = ray.get(namenode.repair_blocks.remote())
print(
    f"NameNode detected {num_dead} dead nodes. Started {jobs_issued} cross-DataNode replication tasks."
)

time.sleep(2)
meta_repaired = ray.get(namenode.get_artifact_metadata.remote("moj_raport.txt"))
node_ids = ray.get([n.get_id.remote() for n in meta_repaired["blocks"][0]["nodes"]])
print(f"Updated nodes containing Block 0 (replication status=2 restored): {node_ids}")


--- FAULT TOLERANCE TEST ---
FAILURE! Corrupting node: DN-3
[Client] Error downloading block 7b21a36c-1080-496e-9e32-98d39fafd227 from node ClientActorHandle(409eaef87e59124ee4a534ed03000000): The actor died unexpectedly before finishing this task.
	class_name: DataNode
	actor_id: 409eaef87e59124ee4a534ed03000000
	pid: 554
	namespace: hdfs-simulation
	ip: 172.20.0.4
The actor is dead because it was killed by `ray.kill`.
[Client] Error downloading block 9ab12ecb-20cc-4eaa-93cc-48e82fea452a from node ClientActorHandle(409eaef87e59124ee4a534ed03000000): The actor died unexpectedly before finishing this task.
	class_name: DataNode
	actor_id: 409eaef87e59124ee4a534ed03000000
	pid: 554
	namespace: hdfs-simulation
	ip: 172.20.0.4
The actor is dead because it was killed by `ray.kill`.
[Client] Error downloading block 4c8f7952-e26b-4d0f-8a5b-7a332f71ef19 from node ClientActorHandle(409eaef87e59124ee4a534ed03000000): The actor died unexpectedly before finishing this task.
	class_name: DataNode


(NameNode pid=489, ip=172.20.0.4) ref.as_future() is deprecated in favor of asyncio.wrap_future(ref.future()).
(NameNode pid=489, ip=172.20.0.4) ref.as_future() is deprecated in favor of asyncio.wrap_future(ref.future()).
(NameNode pid=489, ip=172.20.0.4) ref.as_future() is deprecated in favor of asyncio.wrap_future(ref.future()).
(NameNode pid=489, ip=172.20.0.4) ref.as_future() is deprecated in favor of asyncio.wrap_future(ref.future()).


Updated nodes containing Block 0 (replication status=2 restored): ['DN-1', 'DN-4']


In [9]:
# TEST 4: Cleanup
print("\n--- DELETING EVERYTHING ---")
client.delete_artifact("moj_raport.txt")

time.sleep(1)
val = client.read_artifact("moj_raport.txt")
if val is None:
    print("Caught missing metadata. Artifact deleted, no read")
else:
    print("Found data (Error!)")

for dn in meta_repaired["blocks"][2]["nodes"]:
    if ray.get(dn.get_id.remote()) != unfortunate_node_id:
        blocks = ray.get(dn.list_blocks.remote())
        print(
            f"Amount of old garbage blocks on node {ray.get(dn.get_id.remote())}: empty ({blocks})"
        )


--- DELETING EVERYTHING ---
[Client] File 'moj_raport.txt' completely deleted.
Caught missing metadata. Artifact deleted, no read
Amount of old garbage blocks on node DN-1: empty ([])
Amount of old garbage blocks on node DN-4: empty ([])


In [10]:
# Sprzątanie
ray.shutdown()